<div style="background: linear-gradient(135deg, #0A3D62 0%, #3C6382 100%);
            color: white; padding: 26px 30px; border-radius: 14px;
            font-family: system-ui, sans-serif; margin-bottom: 6px; box-shadow: 0 4px 14px rgba(10,61,98,0.25);">
  <h1 style="margin:0; font-size:1.85rem; font-weight:700;">🔧 Titanic — Feature Engineering</h1>
  <p style="margin:10px 0 0; opacity:0.92;">Leakage-safe transforms (§61.15) · missing-data decisions (§60.17) · quality gates (§60.19)</p>
</div>


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">1. Setup &amp; Load</h2></div>


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"axes.titlecolor": "#0A3D62"})

DATA_DIR = Path("../Datasets")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")

assert train_raw["PassengerId"].is_unique
assert test_raw["PassengerId"].is_unique
assert train_raw["Survived"].isin([0, 1]).all()
print(f"Train {train_raw.shape} | Test {test_raw.shape}")


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;">
  <h2 style="margin:0; font-size:1.25rem;">2. Fit Stats on Train Only (§61.15 / §60.17)</h2>
</div>
<div style="background:#FEF5E7; border-left:5px solid #E67E22; border-radius:0 12px 12px 0; padding:14px 18px; font-family:system-ui,sans-serif;">
  <strong style="color:#D35400;">Leakage control:</strong> Age/Fare/Embarked statistics and rare-title maps are fit on train only, then applied to both sets.
</div>


In [ ]:
def extract_title(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    parts = str(name).split(",")
    if len(parts) < 2:
        return "Unknown"
    return parts[1].strip().split(".")[0].strip()

def map_title(title: str) -> str:
    m = {"Mr": "Mr", "Mrs": "Mrs", "Miss": "Miss", "Master": "Master",
         "Ms": "Miss", "Mlle": "Miss", "Mme": "Mrs",
         "Dr": "Rare", "Rev": "Rare", "Col": "Rare", "Major": "Rare",
         "Capt": "Rare", "Sir": "Rare", "Don": "Rare", "Dona": "Rare",
         "Jonkheer": "Rare", "Lady": "Rare", "Countess": "Rare", "the Countess": "Rare"}
    return m.get(str(title).strip(), "Rare")

def extract_deck(cabin: Any) -> str:
    if pd.isna(cabin) or str(cabin).strip() == "":
        return "Missing"
    return str(cabin).strip()[0].upper()

def fit_stats(train_df: pd.DataFrame) -> dict[str, Any]:
    df = train_df.copy()
    df["Title"] = df["Name"].map(extract_title).map(map_title)
    return {
        "age_by_title_pclass": df.groupby(["Title", "Pclass"])["Age"].median().to_dict(),
        "age_by_title": df.groupby("Title")["Age"].median().to_dict(),
        "age_global": float(df["Age"].median()),
        "fare_by_pclass": df.groupby("Pclass")["Fare"].median().to_dict(),
        "fare_global": float(df["Fare"].median()),
        "embarked_mode": str(df["Embarked"].mode(dropna=True).iloc[0]),
    }

stats = fit_stats(train_raw)
print("Embarked mode:", stats["embarked_mode"])
print("Age global median:", stats["age_global"])
print("Fare by Pclass:", stats["fare_by_pclass"])


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">3. Transform</h2></div>


In [ ]:
def transform(df: pd.DataFrame, stats: dict, ticket_counts: dict) -> pd.DataFrame:
    out = df.copy()
    out["Title"] = out["Name"].map(extract_title).map(map_title)
    out["Deck"] = out["Cabin"].map(extract_deck)
    out["Deck"] = out["Deck"].where(~out["Deck"].isin({"T", "G"}), "Other")
    out["FamilySize"] = out["SibSp"].fillna(0).astype(int) + out["Parch"].fillna(0).astype(int) + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["FamilyBin"] = pd.cut(out["FamilySize"], bins=[0, 1, 4, 20], labels=["Alone", "Small", "Large"]).astype(str)
    out["Embarked"] = out["Embarked"].fillna(stats["embarked_mode"])

    def impute_fare(row):
        if pd.notna(row["Fare"]):
            return float(row["Fare"])
        return float(stats["fare_by_pclass"].get(row["Pclass"], stats["fare_global"]))
    out["Fare"] = out.apply(impute_fare, axis=1)
    out["LogFare"] = np.log1p(out["Fare"])
    out["FareBin"] = pd.cut(out["Fare"], bins=[-0.01, 7.91, 14.45, 31.0, 600], labels=["Q1","Q2","Q3","Q4"]).astype(str)

    def impute_age(row):
        if pd.notna(row["Age"]):
            return float(row["Age"])
        key = (row["Title"], row["Pclass"])
        if key in stats["age_by_title_pclass"] and pd.notna(stats["age_by_title_pclass"][key]):
            return float(stats["age_by_title_pclass"][key])
        if row["Title"] in stats["age_by_title"] and pd.notna(stats["age_by_title"][row["Title"]]):
            return float(stats["age_by_title"][row["Title"]])
        return float(stats["age_global"])
    out["Age"] = out.apply(impute_age, axis=1)
    out["AgeBin"] = pd.cut(out["Age"], bins=[0, 12, 18, 30, 40, 50, 60, 100],
                           labels=["Child","Teen","YoungAdult","Adult","Mid","Senior","Elder"], right=False).astype(str)
    out["TicketGroupSize"] = out["Ticket"].map(ticket_counts).fillna(1).astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    out["SexMale"] = (out["Sex"] == "male").astype(int)
    return out

ticket_counts = pd.concat([train_raw["Ticket"], test_raw["Ticket"]]).value_counts().to_dict()
train_fe = transform(train_raw, stats, ticket_counts)
test_fe = transform(test_raw, stats, ticket_counts)

core = ["Age", "Fare", "Embarked", "Title", "Deck", "FamilySize", "IsAlone", "LogFare"]
assert train_fe[core].isnull().sum().sum() == 0, train_fe[core].isnull().sum()
assert test_fe[core].isnull().sum().sum() == 0
print("Train FE", train_fe.shape, "| Test FE", test_fe.shape)
print("Core features: zero missing ✓")


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">4. Validate Associations After FE</h2></div>


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=train_fe, x="Title", y="Survived", order=["Mrs","Miss","Master","Rare","Mr"],
            errorbar=None, color="#3C6382", ax=axes[0])
axes[0].set_title("Survival rate by Title"); axes[0].set_ylim(0, 1)
sns.barplot(data=train_fe, x="Deck", y="Survived", errorbar=None, color="#0A3D62", ax=axes[1])
axes[1].set_title("Survival rate by Deck"); axes[1].set_ylim(0, 1)
sns.barplot(data=train_fe, x="FamilyBin", y="Survived", hue="FamilyBin",
            order=["Alone","Small","Large"], palette=["#C0392B","#27AE60","#E67E22"],
            legend=False, errorbar=None, ax=axes[2])
axes[2].set_title("Survival rate by FamilyBin"); axes[2].set_ylim(0, 1)
plt.tight_layout(); plt.show()
print(train_fe.groupby("Title")["Survived"].mean().sort_values(ascending=False))


<div style="background:#0A3D62; color:white; padding:12px 20px; border-radius:10px; font-family:system-ui,sans-serif; margin:28px 0 12px;"><h2 style="margin:0; font-size:1.25rem;">5. Persist Artifacts (§60.22)</h2></div>


In [ ]:
train_fe.to_csv(DATA_DIR / "train_fe.csv", index=False)
test_fe.to_csv(DATA_DIR / "test_fe.csv", index=False)

NUM = ["Pclass","Age","SibSp","Parch","Fare","LogFare","FamilySize","IsAlone","TicketGroupSize","HasCabin","SexMale"]
CAT = ["Sex","Embarked","Title","Deck","FamilyBin","AgeBin","FareBin"]
for c in CAT:
    train_fe[c] = train_fe[c].astype(str)
    test_fe[c] = test_fe[c].astype(str)

train_oh = pd.get_dummies(train_fe[CAT], columns=CAT)
test_oh = pd.get_dummies(test_fe[CAT], columns=CAT)
cols = sorted(set(train_oh.columns) | set(test_oh.columns))
train_oh = train_oh.reindex(columns=cols, fill_value=0)
test_oh = test_oh.reindex(columns=cols, fill_value=0)
X_train = pd.concat([train_fe[NUM].reset_index(drop=True), train_oh.reset_index(drop=True)], axis=1)
X_test = pd.concat([test_fe[NUM].reset_index(drop=True), test_oh.reset_index(drop=True)], axis=1)
y = train_fe["Survived"].astype(int)
X_train.to_csv(DATA_DIR / "X_train_oh.csv", index=False)
X_test.to_csv(DATA_DIR / "X_test_oh.csv", index=False)
y.to_csv(DATA_DIR / "y_train.csv", index=False, header=["Survived"])
print("Saved train_fe, test_fe, X_train_oh, X_test_oh, y_train")
print("X_train", X_train.shape, "X_test", X_test.shape)


<div style="background:#EAF7F0; border-left:5px solid #27AE60; border-radius:0 12px 12px 0; padding:16px 20px; font-family:system-ui,sans-serif;">
  <strong style="color:#1E8449;">Summary</strong>
  <ul style="margin:8px 0 0; padding-left:1.2rem; color:#1B262C; line-height:1.55;">
    <li>Title, Deck, FamilySize/IsAlone/FamilyBin, LogFare, Age/FareBins, TicketGroupSize, HasCabin</li>
    <li>Age: conditional median (Title×Pclass) — MAR-oriented treatment</li>
    <li>Cabin absence kept as Missing/Deck — informative missingness</li>
    <li>All impute stats from train only · zero missing in core features</li>
  </ul>
</div>
<div style="background:linear-gradient(135deg,#F8FBFD 0%,#E8F1F5 100%); border:1px solid #B8D0DC; border-radius:12px; padding:16px 20px; font-family:system-ui,sans-serif; margin-top:12px;">
  <strong style="color:#0A3D62;">Next:</strong> <code>03_baseline_models.ipynb</code>
</div>
